라이브러리 import

In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
mpl.rc('font', family = 'Malgun Gothic')
import matplotlib.pyplot as plt
import datetime
import math
import openpyxl

import plotly.graph_objects as go # 그래프 그릴 때 사용할 라이브러리입니다.
from plotly.subplots import make_subplots

# 투자전략

> 배당수익율 8% 이상

> 2년간 DPS가 상승한기업

중
> 당기순이익증가율이 가장 큰 10개 기업


## Parameter 조정
백테스트를 하기 위한 기본 파라미터 조정

In [2]:
# 리밸런싱 조건 설정해주기
# 리밸런싱 주기
rebalancing = 365 #일 단위

# 거래비용
tax = 0.0002 # 현재 거래비용은 0.02%입니다.

# 투자원금
cash = 100000000 # 모의투자 시작금액인 1억으로 시작하겠습니다.
money = cash

risk_free_rate = 0.0253 / 365 # 25년 9월 1일 기준 CD91일물 금리로 하겠습니다.

# 손절기준
loss_cut = 0.1 #매수 이후 10% 이상 가격이 떨어지면 매도하겠습니다.

day = 0
quarter = 0
df_list = '현금 주식총액 포트폴리오가치 일일수익률 총수익률'.split() # 백테스트 데이터프레임에 보여줄 항목들 입니다. 이후에 더 추가할 예정입니다.
backtest = pd.DataFrame(columns = df_list)

## 데이터 로드

In [3]:
#상장폐지된 주가 데이터까지 모두 받아오기 -> 편향을 제거하기 위해서입니다.
price_data = pd.read_excel('백테스트.xlsx', index_col = 0, sheet_name="수정주가")

div = pd.read_excel('백테스트.xlsx', index_col = 0, sheet_name = '배당수익률' )

dps = pd.read_excel('백테스트.xlsx', index_col = 0,  sheet_name = 'DPS')

NI = pd.read_excel('백테스트.xlsx', index_col = 0,  sheet_name = '당기순이익')

# 리밸런싱 주기와 데이터의 날짜가 맞지않을 경우 코드를 짜기 까다로우니 주의해주세요!!

In [4]:
print(price_data.shape, div.shape, dps.shape, NI.shape)

(3654, 820) (10, 820) (10, 820) (10, 820)


# 스크리닝 & 백테스트

In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [19]:
# Initialize variables for backtest start (moved from earlier cell for re-run robustness)
day = 0
quarter = 0
money = cash # Reset money to initial cash
backtest = pd.DataFrame(columns = df_list) # Reset backtest DataFrame

print('전체 리밸런싱 횟수는 {}'.format(int(len(price_data)/rebalancing)))
for reb in range(int(len(price_data)/rebalancing)): #리밸런싱 횟수를 계산해줍니다. 전체 투자기간/리밸런싱 주기
    print( reb+1, '회 투자')
    print('현금 : ', money)
    inv = price_data.iloc[day:day+rebalancing,:] #전체 주가 데이터에서 투자기간동안의 데이터를 뽑아냅니다.
    inv = inv.replace(np.nan, 0) # 수정주가 데이터에 없는 값들은 0으로 처리해줍니다.
    inv = inv.loc[:,inv.iloc[0,:] > 0 ] # 투자 시작 시점에 데이터가 0인 친구들은 다 날려줍니다.

    ## 스크리닝

    ### 조건 1. 배당수익률 > 6 완

    try:
      div_inv = div.iloc[quarter, :] # 투자 시점의 데이터를 불러와줍니다.
      div_inv = div_inv.dropna() # 투자 시점에 없는 데이터들은 제거해줍니다.
      div_inv = div_inv[div_inv>6] #
      div_inv.dropna(inplace=True)
      div_inv_list = div_inv.index # 리스트로 저장해줍니다.

 ### 조건 2. dps 2년간 상승 기업 완
      dps_inv = pd.DataFrame(dps.iloc[:quarter, :])

      dps_diff_inv = dps_inv.diff()
      dps_diff_inv = dps_diff_inv.iloc[-2:,:]
      dps_diff_inv = dps_diff_inv[dps_diff_inv>0] # 차분된 값 중 0이하를 NaN으로 바꿈
      dps_diff_inv = dps_diff_inv.dropna(axis=1)
      dps_inv_list = dps_diff_inv.columns #리스트로 저장해줍니다.


      ### 조건 1,2 충족 기업 중 NI상승률 상위 10개

      constraint = div_inv_list.intersection(dps_inv_list) # 두 조건 모두에 걸리는 종목들을 선정해줍니다.
      NI_growth = NI[constraint]
      NI_growth = NI_growth.pct_change()
      NI_growth = NI_growth.iloc[quarter,:]

      inv_list = NI_growth.sort_values(ascending=False).head(5).index
    except:
      inv_list = []

    print('투자 후보 갯수는 : ', len(inv_list))

    ## 여기까지 스크리닝 과정

    final_inv_list = []
    for i in range(len(inv_list)):
        if inv_list[i] in inv.columns:
            final_inv_list.append(inv_list[i])
        else:
            print(inv_list[i],' 종목이 없습니다')
    print('투자하는 종목의 수는 : ', len(final_inv_list))
    print('투자종목 : ',final_inv_list)
    # 스크리닝 된 종목 들 중 가격 데이터에 있는 종목들만 투자 리스트에 추가해줍니다.


    # 매수 기준 : 동일 비중
    # 보유한 주식의 가치 평가하기

    if len(final_inv_list)==0:
        allocation=0
    else:
        allocation = money / len(final_inv_list) # 동일 비중이기 때문에 보유 현금 / 투자할 종목 수 로 나눠줍니다.


    print('동일 비중 투자 금액은 : ' , allocation) # 저는 제 코드를 믿지 못하기 때문에 이렇게 중간중간 어디서 에러가 나는지 예상할 수 있게 출력 결과도 함께 프린트합니다.
    final_price_data = inv[final_inv_list].copy()

    vec = pd.DataFrame({'매수수량' : allocation // final_price_data.iloc[0,:]}) # 종목당 매수할 수를 데이터프레임(벡터) 로 만들어줍니다.
    vec = vec.replace(np.nan, 0)


    # 매도(손절 기준) : 10% 이상 떨어질 경우 매도

    # 손절을 할 때 보유하고 있는 주식수를 0으로 만들거나
    # 손절 이후 포트폴리오의 가격을 0으로 만들거나의 방법을 선택할 수 있겠죠?
    # 저는 손절 한 순간부터 포트폴리오에서 주가를 계산하지 않아도 되는 방법이 더 편할 것 같아서 이렇게 선택했지만
    # 저와 다르게 계산하셔도 상관없습니다.
    loss_cut_money_list = []
    loss_cut_money = 0
    for days in final_price_data.index:
        for stocks in final_price_data.columns:
            if final_price_data.loc[days, stocks] < final_price_data[stocks][0] * ( 1 - loss_cut ):
                loss_cut_money = loss_cut_money + (final_price_data.loc[days, stocks] * float(vec.loc[stocks])) * (1 - tax)
                final_price_data.loc[days:, stocks] = 0

        loss_cut_money_list.append(loss_cut_money)


        # 날짜에 저장된 값을 리스트에 추가해줍니다. 이 반복문을 돌면서 포트폴리오 투자기간동안의 현금이 저장될 것입니다.


    product = np.dot(final_price_data, vec) #np.dot 통해 행렬과 벡터의 내적값을 구해줍니다.
    product = pd.DataFrame(product) #데이터프레임으로 만들어서 나중에 붙여주겠습니다.


    balance = pd.DataFrame(index = product.index)
    balance['현금'] = money - product.iloc[0,0] # 현금은 리밸런싱이 이루어지는 날의 현금에서 주식 매수대금을 뺀 값입니다.
    balance['현금'] += loss_cut_money_list # 손절한 금액은 날짜별로 더해줍시다


    _backtest = pd.DataFrame(columns = df_list) # 빈 데이터프레임을 만들어줍니다.
    _backtest['주식총액'] = product # 보유하고 있는 주식의 가치입니다.
    _backtest['현금'] = balance['현금'] # 아까 계산한 일자별 현금 데이터를 넣어줍니다.


    _backtest['포트폴리오가치'] = _backtest['현금'] + _backtest['주식총액'] # 포트폴리오 가치는 현금과 주식총액의 합입니다.
    _backtest['총수익률'] = (_backtest['포트폴리오가치']/cash) # 처음 들고 시작한 돈 대비 수익률
    _backtest.index = price_data.index[day : day + rebalancing]


    backtest = pd.concat([backtest, _backtest], axis = 0, ignore_index=False)
    # 데이터프레임들을 계속 붙여주면서 백테스트 데이터를 만들겠습니다.
    # 반복문을 돌면서 빈데이터프레임에 데이터를 채워가기 때문에 먼저 선언해준 백테스트 데이터프레임에 붙여줍니다.


    # 리밸런싱 이후 시작 금액은 주식 판매대금 - 거래비용 + 현금입니다.
    money = backtest.iloc[-1,0] + (backtest.iloc[-1,1] * ( 1 - tax))
    day = day + rebalancing
    quarter += 1

backtest['일일수익률'] = backtest['포트폴리오가치'].pct_change()

전체 리밸런싱 횟수는 10
1 회 투자
현금 :  100000000
투자 후보 갯수는 :  5
투자하는 종목의 수는 :  5
투자종목 :  ['A015760', 'A071320', 'A001270', 'A008110', 'A003780']
동일 비중 투자 금액은 :  20000000.0
2 회 투자
현금 :  107910772.88
투자 후보 갯수는 :  0
투자하는 종목의 수는 :  0
투자종목 :  []
동일 비중 투자 금액은 :  0
3 회 투자
현금 :  107910772.88
투자 후보 갯수는 :  0
투자하는 종목의 수는 :  0
투자종목 :  []
동일 비중 투자 금액은 :  0
4 회 투자
현금 :  107910772.88
투자 후보 갯수는 :  3
투자하는 종목의 수는 :  3
투자종목 :  ['A002460', 'A005930', 'A000650']
동일 비중 투자 금액은 :  35970257.626666665
5 회 투자
현금 :  96248627.84
투자 후보 갯수는 :  2
투자하는 종목의 수는 :  2
투자종목 :  ['A084670', 'A003540']
동일 비중 투자 금액은 :  48124313.92
6 회 투자
현금 :  86160045.83000001
투자 후보 갯수는 :  5
투자하는 종목의 수는 :  5
투자종목 :  ['A003540', 'A100250', 'A122900', 'A175330', 'A007340']
동일 비중 투자 금액은 :  17232009.166
7 회 투자
현금 :  145583270.923
투자 후보 갯수는 :  5
투자하는 종목의 수는 :  5
투자종목 :  ['A298020', 'A003540', 'A011780', 'A005960', 'A071050']
동일 비중 투자 금액은 :  29116654.184600003
8 회 투자
현금 :  130183201.523
투자 후보 갯수는 :  5
투자하는 종목의 수는 :  5
투자종목 :  ['A001450', 'A005830', 'A000810',

## 벤치마크
코스피 지수를 벤치마크로 설정

In [9]:
BM = pd.read_excel('KOSPI.xlsx',index_col=0)
BM['일일수익률'] = BM['시가'].pct_change()
BM['총수익률'] = BM['시가']/BM['시가'][0]
BM = BM.dropna(axis=0)
BM

,시가,일일수익률,총수익률
날짜,,,
2015-09-18,1978.39,-0.005369,0.994631
2015-09-19,1978.39,0.000000,0.994631
2015-09-20,1978.39,0.000000,0.994631
2015-09-21,1973.24,-0.002603,0.992042
2015-09-22,1973.54,0.000152,0.992192
...,...,...,...
2025-09-13,3374.65,0.000000,1.696597
2025-09-14,3374.65,0.000000,1.696597
2025-09-15,3407.78,0.009817,1.713253


## 성과지표

> CAGR: 연평균 성장률 ( Compound Annual Growth Rate) => (최종가치 - 시작가치) * (1/전체년수) - 1

In [22]:
## CAGR

num_of_year = int(len(backtest)/365)

CAGR = (backtest['포트폴리오가치'][-1]/cash)**(1/num_of_year) - 1
print('포트폴리오 CAGR은 {} %'.format(CAGR*100))

BENCHMARK_CAGR = ((BM['시가'][3910] / BM['시가'][0])) ** (1 / num_of_year) - 1
print("벤치마크 CAGR은 {} %".format(BENCHMARK_CAGR*100))

포트폴리오 CAGR은 8.712341234703903 %


IndexError: index 3910 is out of bounds for axis 0 with size 3653

> MDD: 최저점이 최고점 대비 몇%의 하락인지? => (최고점 - 최저점) / 최고점

In [23]:
## MDD

max_list = [backtest.iloc[0,2]]
min_list = [backtest.iloc[0,2]]

for i in range(len(backtest)):
    if i==0:
        max_list.append(backtest.iloc[0,2])
        min_list.append(backtest.iloc[0,2])

    else:
        if backtest.iloc[i, 2] >= backtest.iloc[i-1,2]:
            max_list.append(backtest.iloc[i, 2])
            min_list.append(backtest.iloc[i, 2])
        else:
            if(max_list[-1]>backtest.iloc[:i,2].max()):
                max_list.append(max_list[-1])
            else:
                max_list.append(backtest.iloc[:i,2].max())
            min_list.append(backtest.iloc[i, 2])


max_list = max_list[1:]
min_list = min_list[1:]
backtest['max'] = max_list
backtest['min'] = min_list
backtest['mdd'] = -((backtest['max'] - backtest['min'])/backtest['max'])

print('BACKTEST MDD는 {}%'.format(backtest['mdd'].min()*100))

backtest['MDD'] = 0
for i in range(len(backtest)):
    if i != 0:
        backtest.iloc[i, -1] = backtest['mdd'][:i].min()


# benchmark mdd
BENCHMARK_max_list = [BM.iloc[0,0]]
BENCHMARK_min_list = [BM.iloc[0,0]]

for i in range(len(BM)):
    if i==0:
        BENCHMARK_max_list.append(BM.iloc[0,0])
        BENCHMARK_min_list.append(BM.iloc[0,0])

    else:
        if BM.iloc[i, 2] >= BM.iloc[i-1,2]:
            BENCHMARK_max_list.append(BM.iloc[i, 0])
            BENCHMARK_min_list.append(BM.iloc[i, 0])
        else:
            if(BENCHMARK_max_list[-1]>BM.iloc[:i,2].max()):
                BENCHMARK_max_list.append(BENCHMARK_max_list[-1])
            else:
                BENCHMARK_max_list.append(backtest.iloc[:i,2].max())
            BENCHMARK_min_list.append(BM.iloc[i, 0])


BENCHMARK_max_list = BENCHMARK_max_list[1:]
BENCHMARK_min_list = BENCHMARK_min_list[1:]
BM['max'] = BENCHMARK_max_list
BM['min'] = BENCHMARK_min_list
BM['mdd'] = -((BM['max'] - BM['min'])/BM['max'])

print('BENCHMARK MDD는 {}%'.format(BM['mdd'].min()*100))

BM['MDD'] = 0
for i in range(len(BM)):
    if i != 0:
        BM.iloc[i, -1] = BM['mdd'][:i].min()

BACKTEST MDD는 -33.28336609529593%
BENCHMARK MDD는 -12.36168838106091%


> Sharpe Ratio: 위험(변동성) 대비 초과수익을 얼마나 냈는지? => (자산 X의 기대수익률 – 무위험 자산 수익률) / 자산 X의 기대수익률의 표준편차

In [24]:
## Sharpe Ratio
backtest_return = backtest['일일수익률'].mean()
backtest_std = backtest['일일수익률'].std()

BM_return = BM['일일수익률'].mean()
BM_std = BM['일일수익률'].std()

backtest_sharpe = (backtest_return - risk_free_rate) / backtest_std
BM_sharpe = (BM_return - risk_free_rate) / BM_std

print('BACKTEST 샤프 비율은 : ', backtest_sharpe)
print('BENCHMARK 샤프 비율은 : ', BM_sharpe)

BACKTEST 샤프 비율은 :  0.026600505888641567
BENCHMARK 샤프 비율은 :  0.013589718064301908


> Sortino Ratio: 하락 변동성 대비 수익을 얼마나 냈는지? => (자산 X의 기대수익률 – 무위험 자산 수익률) / 자산 X의 손실구간(수익률<0)의 표준편차

In [25]:
## Sortino Ratio
backtest_return = backtest['일일수익률'].mean()
BM_return = BM['일일수익률'].mean()

backtest_negative = backtest['일일수익률'][backtest['일일수익률'] < 0]
BM_negative = BM['일일수익률'][BM['일일수익률'] < 0]

backtest_downside_std = np.std(backtest_negative, ddof=1)  # 표본표준편차
BM_downside_std = np.std(BM_negative, ddof=1)

backtest_sortino = (backtest_return - risk_free_rate) / backtest_downside_std if backtest_downside_std != 0 else np.nan
BM_sortino = (BM_return - risk_free_rate) / BM_downside_std if BM_downside_std != 0 else np.nan

print('BACKTEST 소르티노 비율은 : ', backtest_sortino)
print('BENCHMARK 소르티노 비율은 : ', BM_sortino)

BACKTEST 소르티노 비율은 :  0.022768342875931002
BENCHMARK 소르티노 비율은 :  0.014483692200018073


### 백테스트 결과 시각화

In [26]:
backtest

,현금,주식총액,포트폴리오가치,일일수익률,총수익률,max,min,mdd,MDD
2015-09-17,9.674000e+04,99903260.0,1.000000e+08,NaN,1.000000,1.000000e+08,1.000000e+08,-0.000000,0.000000
2015-09-18,9.674000e+04,99696820.0,9.979356e+07,-0.002064,0.997936,1.000000e+08,9.979356e+07,-0.002064,0.000000
2015-09-19,9.674000e+04,99696820.0,9.979356e+07,0.000000,0.997936,9.979356e+07,9.979356e+07,-0.000000,-0.002064
2015-09-20,9.674000e+04,99696820.0,9.979356e+07,0.000000,0.997936,9.979356e+07,9.979356e+07,-0.000000,-0.002064
2015-09-21,9.674000e+04,100110630.0,1.002074e+08,0.004147,1.002074,1.002074e+08,1.002074e+08,-0.000000,-0.002064
...,...,...,...,...,...,...,...,...,...
2025-09-09,3.108530e+07,197265420.0,2.283507e+08,0.007133,2.283507,2.283507e+08,2.283507e+08,-0.000000,-0.332834
2025-09-10,3.108530e+07,198929250.0,2.300146e+08,0.007286,2.300146,2.300146e+08,2.300146e+08,-0.000000,-0.332834
2025-09-11,3.108530e+07,199974180.0,2.310595e+08,0.004543,2.310595,2.310595e+08,2.310595e+08,-0.000000,-0.332834
2025-09-12,3.108530e+07,199477100.0,2.305624e+08,-0.002151,2.305624,2.310595e+08,2.305624e+08,-0.002151,-0.332834


In [27]:
backtest.iloc[:i,2].max()

231059482.185

In [28]:
fig = make_subplots(rows=4, cols=1,
                    specs=[[{"rowspan":3}],
                          [None],
                          [None],
                          [{}]],
                   shared_xaxes=True,
                   vertical_spacing=0.2,
                   subplot_titles=("수익률","MDD"))

fig.add_trace(go.Scatter(name='코스피 총수익률', x=BM.index, y=BM['총수익률']),
             row=1, col=1)
fig.add_trace(go.Scatter(name='포트폴리오 총수익률', x=backtest.index, y=backtest['총수익률']),
             row=1, col=1)
fig.add_trace(go.Scatter(name='포트폴리오 MDD', x=backtest.index, y=backtest['MDD'], fill='tozeroy'),
             row=4, col=1)
fig.add_trace(go.Scatter(name='BENCHMARK MDD', x=BM.index, y=BM['MDD'], fill='tozeroy'),
             row=4, col=1)

fig.update_layout(height=800, width=1000, plot_bgcolor='rgb(240, 240,240)',
                 title_text="백테스트 결과")

fig.update_layout(
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1,
                     label="1m",
                     step="month",
                     stepmode="backward"),
                dict(count=6,
                     label="6m",
                     step="month",
                     stepmode="backward"),
                dict(count=1,
                     label="YTD",
                     step="year",
                     stepmode="todate"),
                dict(count=1,
                     label="1y",
                     step="year",
                     stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    )
)

fig.show()